Primera parte del exploratorio, lo que quiero ver es como unir los 3 excels juntos y cuan mal están:

In [1]:
from datetime import datetime, time, timedelta
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXCEL_DIR = ROOT / "data" / "excels"
archivos = sorted(EXCEL_DIR.glob("Totalizadores Planta *.xlsx"))

HOJA_REF = "Consolidado EE"   # hoja con el target (Frio)
LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche


def a_segundos(v):
    """Convierte una celda de hora a segundos desde las 00:00, o None si no se entiende."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    if isinstance(v, datetime):
        v = v.time()
    if isinstance(v, time):
        return v.hour * 3600 + v.minute * 60 + v.second
    if isinstance(v, timedelta):
        return int(v.total_seconds())
    if isinstance(v, (int, float)):          # Excel guarda horas como fracción de día
        return round(v * 86400)
    if isinstance(v, str):
        partes = v.strip().split(":")
        if len(partes) == 3 and all(p.isdigit() for p in partes):
            h, m, s = map(int, partes)
            return h * 3600 + m * 60 + s
    return None                               # formato raro, como "0:1:00:01"


def formato(seg):
    return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"


for archivo in archivos:
    print(f"\n=== {archivo.name} ===")
    df = pd.read_excel(archivo, sheet_name=HOJA_REF, engine="openpyxl")

    horas = df.iloc[:, 1]                     # segunda columna: la hora
    segundos = horas.map(a_segundos)

    # 1) Horas válidas posteriores a las 23:00:00 (sin incluirla)
    tarde = segundos[segundos > LIMITE]
    print("Horas > 23:00:00 (valor: cantidad de filas)")
    for seg, n in tarde.value_counts().sort_index().items():
        print(f"  {formato(int(seg))}: {n}")

    # 2) Valores que no pude interpretar como hora
    raros = horas[segundos.isna() & horas.notna()]
    if len(raros):
        print("Valores con formato no reconocido:")
        for valor, n in raros.astype(str).value_counts().items():
            print(f"  {valor!r}: {n}")


=== Totalizadores Planta 2020_2022.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 10
  23:59:00: 614
  23:59:01: 7
  23:59:59: 1

=== Totalizadores Planta 2021_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 9
  23:59:00: 605
  23:59:01: 5
  23:59:59: 1

=== Totalizadores Planta 2022_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 5
  23:59:00: 475
  23:59:01: 3
  23:59:59: 1


In [7]:
HOJA_SEG = "Seguimiento Dia"

LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche
HOJA_SEG = "Seguimiento Dia"
FIN_DIA = 23 * 3600 + 59 * 60   # 23:59:00

def a_segundos(v):
    """Convierte una celda de hora a segundos desde las 00:00, o None si no se entiende."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    if isinstance(v, datetime):
        v = v.time()
    if isinstance(v, time):
        return v.hour * 3600 + v.minute * 60 + v.second
    if isinstance(v, timedelta):
        return int(v.total_seconds())
    if isinstance(v, (int, float)):          # Excel guarda horas como fracción de día
        return round(v * 86400)
    if isinstance(v, str):
        partes = v.strip().split(":")
        if len(partes) == 3 and all(p.isdigit() for p in partes):
            h, m, s = map(int, partes)
            return h * 3600 + m * 60 + s
    return None                               # formato raro, como "0:1:00:01"

FIN_DIA = 23 * 3600 + 59 * 60   # 23:59:00


def a_diario(df):
    """Pasa una hoja horaria a una fila por día.

    df : hoja con DIA en la 1ª columna y hora en la 2ª.
    Devuelve una fila por fecha, con columnas extra:
      seg_elegido : hora elegida (segundos desde 00:00)
      fuente      : '23:59' o 'ultima_hora' (qué regla se usó)
      n_lecturas  : cuántas filas tenía ese día
    """
    fecha = pd.to_datetime(df.iloc[:, 0], errors="coerce").dt.normalize()
    segs = df.iloc[:, 1].map(a_segundos)
    d = df.iloc[:, 2:].copy()
    d.insert(0, "seg", segs)
    d.insert(0, "fecha", fecha)

    # Descarto filas sin fecha u hora interpretables (ej. "0:1:00:01")
    invalidas = d["fecha"].isna() | d["seg"].isna()
    if invalidas.any():
        print(f"  Filas descartadas por fecha/hora no válida: {int(invalidas.sum())}")
    d = d[~invalidas].copy()

    # Cantidad de lecturas por día (antes de quedarme con una)
    d["n_lecturas"] = d.groupby("fecha")["seg"].transform("size")

    # Fuente: 23:59 o después, o cualquier otra hora
    d["fuente"] = "ultima_hora"
    d.loc[d["seg"] >= FIN_DIA, "fuente"] = "23:59"

    # Por día: primero las de 23:59, y entre iguales la hora más tardía
    d["_prio"] = (d["fuente"] != "23:59").astype(int)   # 0 = 23:59, 1 = otra
    d = d.sort_values(["fecha", "_prio", "seg"], ascending=[True, True, False])
    dia = d.groupby("fecha", as_index=False).head(1)

    dia = dia.rename(columns={"seg": "seg_elegido"}).drop(columns="_prio")
    return dia.sort_values("fecha").reset_index(drop=True)

def a_diario(df):
    """Pasa una hoja horaria a una fila por día.

    df : hoja con DIA en la 1ª columna y hora en la 2ª.
    Devuelve una fila por fecha, con columnas extra:
      seg_elegido : hora elegida (segundos desde 00:00)
      fuente      : '23:59' o 'ultima_hora' (qué regla se usó)
      n_lecturas  : cuántas filas tenía ese día
    """
    fecha = pd.to_datetime(df.iloc[:, 0], errors="coerce").dt.normalize()
    segs = df.iloc[:, 1].map(a_segundos)
    d = df.iloc[:, 2:].copy()
    d.insert(0, "seg", segs)
    d.insert(0, "fecha", fecha)

    # Descarto filas sin fecha u hora interpretables (ej. "0:1:00:01")
    invalidas = d["fecha"].isna() | d["seg"].isna()
    if invalidas.any():
        print(f"  Filas descartadas por fecha/hora no válida: {int(invalidas.sum())}")
    d = d[~invalidas].copy()

    # Cantidad de lecturas por día (antes de quedarme con una)
    d["n_lecturas"] = d.groupby("fecha")["seg"].transform("size")

    # Fuente: 23:59 o después, o cualquier otra hora
    d["fuente"] = "ultima_hora"
    d.loc[d["seg"] >= FIN_DIA, "fuente"] = "23:59"

    # Por día: primero las de 23:59, y entre iguales la hora más tardía
    d["_prio"] = (d["fuente"] != "23:59").astype(int)   # 0 = 23:59, 1 = otra
    d = d.sort_values(["fecha", "_prio", "seg"], ascending=[True, True, False])
    dia = d.groupby("fecha", as_index=False).head(1)

    dia = dia.rename(columns={"seg": "seg_elegido"}).drop(columns="_prio")
    return dia.sort_values("fecha").reset_index(drop=True)

In [8]:
diarios = {}
for archivo in archivos:
    print(f"\n=== {archivo.name} ===")
    df_ee = pd.read_excel(archivo, sheet_name=HOJA_REF, engine="openpyxl")

    dia = a_diario(df_ee)
    diarios[archivo.name] = dia

    print(f"  Días: {len(dia)}  ({dia['fecha'].min().date()} -> {dia['fecha'].max().date()})")
    print("  Fuente usada:", dia["fuente"].value_counts().to_dict())
    print("  Días con < 24 lecturas:", int((dia["n_lecturas"] < 24).sum()))
    print("  Días con > 25 lecturas:", int((dia["n_lecturas"] > 25).sum()))


=== Totalizadores Planta 2020_2022.xlsx ===
  Días: 612  (2020-07-01 -> 2022-03-16)
  Fuente usada: {'23:59': 609, 'ultima_hora': 3}
  Días con < 24 lecturas: 3
  Días con > 25 lecturas: 8

=== Totalizadores Planta 2021_2023.xlsx ===
  Días: 602  (2021-07-01 -> 2023-03-06)
  Fuente usada: {'23:59': 599, 'ultima_hora': 3}
  Días con < 24 lecturas: 5
  Días con > 25 lecturas: 8

=== Totalizadores Planta 2022_2023.xlsx ===
  Días: 474  (2022-07-01 -> 2023-10-26)
  Fuente usada: {'23:59': 471, 'ultima_hora': 3}
  Días con < 24 lecturas: 5
  Días con > 25 lecturas: 6
